# Customer Support Triage — Data Preprocessing

This notebook prepares the English customer support ticket dataset for machine learning.

The preprocessing pipeline includes:

1. Loading the dataset
2. Selecting relevant columns
3. Handling missing textual values
4. Creating the final ticket text
5. Checking empty records
6. Preparing target labels
7. Investigating repeated ticket text
8. Preventing data leakage during dataset splitting
9. Creating training, validation, and test datasets

The processed data will be used for the subsequent TF-IDF baseline and transformer-based modeling.

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/customer_support_tickets.csv")

In [4]:
df.shape

(61765, 16)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 61765 entries, 0 to 61764
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   subject   56466 non-null  str    
 1   body      61763 non-null  str    
 2   answer    48576 non-null  str    
 3   type      48587 non-null  str    
 4   queue     61765 non-null  str    
 5   priority  61765 non-null  str    
 6   language  61765 non-null  str    
 7   version   28587 non-null  float64
 8   tag_1     48587 non-null  str    
 9   tag_2     48528 non-null  str    
 10  tag_3     48356 non-null  str    
 11  tag_4     43990 non-null  str    
 12  tag_5     27636 non-null  str    
 13  tag_6     13225 non-null  str    
 14  tag_7     5968 non-null   str    
 15  tag_8     2472 non-null   str    
dtypes: float64(1), str(15)
memory usage: 57.2 MB


In [6]:
df = df[df["language"] == "en"].copy()
df.shape

(28261, 16)

In [7]:
df_processed = df.copy()

In [8]:
model_columns = [
    "subject",
    "body",
    "type",
    "priority",
    "queue"
]

df_processed = df_processed[model_columns].copy()

In [9]:
df_processed.shape

(28261, 5)

In [10]:

df_processed.head()

,subject,body,type,priority,queue
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",Incident,high,Technical Support
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Request,medium,Returns and Exchanges
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Request,low,Billing and Payments
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Problem,medium,Sales and Pre-Sales
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Request,high,Technical Support


In [11]:
#checking for the missing values
df_processed.isna().sum()

subject     3639
body           1
type           0
priority       0
queue          0
dtype: int64

## 2.1 Handling Missing Text

The `subject` column contains 3,639 missing values, while the `body` column contains only 1 missing value.

Since the subject and body are the primary textual inputs to the model, these missing values will be replaced with empty strings rather than removing the records.

This preserves potentially useful information from the remaining text field. For example, a ticket with a missing subject may still contain a complete and informative body.

The target columns (`type`, `priority`, and `queue`) contain no missing values and therefore require no missing-value treatment.

In [12]:
df_processed["subject"] = df_processed["subject"].fillna("")
df_processed["body"] = df_processed["body"].fillna("")

In [13]:
df_processed.isna().sum()

subject     0
body        0
type        0
priority    0
queue       0
dtype: int64

## 2.2 Creating the Input Text

The model will use both the ticket subject and the ticket body as input.

The subject provides a concise summary of the issue, while the body contains additional context and details.

These two fields are therefore combined into a single `text` feature.

The original `subject` and `body` columns are retained for analysis and debugging, while `text` will be used as the primary input to the NLP models.

In [14]:
df_processed["text"] = (
    df_processed["subject"].str.strip()
    + " "
    + df_processed["body"].str.strip()
)

In [15]:
df_processed[["subject", "body", "text"]].head()

,subject,body,text
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Account Disruption Dear Customer Support Team,..."
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Query About Smart Home System Integration Feat...
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Inquiry Regarding Invoice Details Dear Custome...
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Question About Marketing Agency Software Compa...
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...","Feature Query Dear Customer Support,\n\nI hope..."


In [16]:
empty_text = df_processed["text"].str.strip().eq("").sum()

print("Completely empty tickets:", empty_text)

Completely empty tickets: 0


## 2.3 Investigating Repeated Ticket Text

During EDA, 4,513 records were identified with repeated `subject` and `body` combinations.

Before creating the train, validation, and test sets, these repeated text records need to be investigated.

If the same ticket text appears in both the training and test sets, the model may effectively see the same example during training and evaluation. This can lead to data leakage and an overly optimistic estimate of model performance.

Therefore, repeated text groups are analyzed to determine whether they have consistent target labels.

In [17]:
duplicate_text = (
    df_processed
    .groupby("text")
    .agg(
        count=("text", "size"),
        type_unique=("type", "nunique"),
        priority_unique=("priority", "nunique"),
        queue_unique=("queue", "nunique")
    )
)

duplicate_text = duplicate_text[
    duplicate_text["count"] > 1
]

print("Repeated text groups:", len(duplicate_text))

Repeated text groups: 4513


In [18]:
print(
    "Repeated text groups with different type labels:",
    (duplicate_text["type_unique"] > 1).sum()
)

print(
    "Repeated text groups with different priority labels:",
    (duplicate_text["priority_unique"] > 1).sum()
)

print(
    "Repeated text groups with different queue labels:",
    (duplicate_text["queue_unique"] > 1).sum()
)

Repeated text groups with different type labels: 0
Repeated text groups with different priority labels: 0
Repeated text groups with different queue labels: 0


## 2.4 Repeated Text Handling

The dataset contains 4,513 repeated text groups based on the combined `subject` and `body` fields.

An investigation was performed to determine whether repeated text was associated with inconsistent labels.

The analysis found:

- 0 repeated text groups with different `type` labels
- 0 repeated text groups with different `priority` labels
- 0 repeated text groups with different `queue` labels

Therefore, repeated ticket text has consistent labels in the dataset.

The repeated records will not be removed solely because they have identical text. Instead, they will be considered when creating the train, validation, and test sets.

All occurrences of the same text should remain within the same dataset split. This prevents the same text from appearing in both training and evaluation sets and reduces the risk of data leakage.

This approach allows repeated examples to remain available for model training while ensuring that evaluation is performed on text that was not directly seen during training.

## 2.5 Text Cleaning and Normalization

The ticket text contains formatting artifacts such as HTML tags, newline characters,
extra whitespace, URLs, and email addresses.

In this step, the text will be normalized while preserving the important semantic
information needed for ticket classification.

We will keep the original `text` column unchanged and create a separate
`clean_text` column. This allows us to compare the original and processed text
and prevents accidental loss of information.

In [19]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Convert HTML line breaks and tags into spaces
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)

    # Replace URLs with a placeholder
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)

    # Replace email addresses with a placeholder
    text = re.sub(r"\S+@\S+", " EMAIL ", text)

    # Replace escaped newline/tab characters
    text = re.sub(r"\\n|\\t", " ", text)

    # Convert actual newline/tab characters to spaces
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Convert multiple spaces to one
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    # Lowercase
    text = text.lower()

    return text

In [20]:
df_processed["clean_text"] = df_processed["text"].apply(clean_text)

In [21]:
df_processed[["text", "clean_text"]].head()

,text,clean_text
1,"Account Disruption Dear Customer Support Team,...","account disruption dear customer support team,..."
2,Query About Smart Home System Integration Feat...,query about smart home system integration feat...
3,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...
4,Question About Marketing Agency Software Compa...,question about marketing agency software compa...
5,"Feature Query Dear Customer Support,\n\nI hope...","feature query dear customer support, i hope th..."


In [22]:
print("Original example:")
print(df_processed["text"].iloc[0])

print("\nCleaned example:")
print(df_processed["clean_text"].iloc[0])

Original example:
Account Disruption Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?

Cleaned example:
account disruption dear customer support team, i am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. this outage is blocking access to account settings, leading to substantial inconvenience. i have attempted to log in multiple times using different browsers and devices, but the issue persists. could you please provide an

In [23]:
empty_clean_text = df_processed["clean_text"].str.strip().eq("").sum()

print("Empty clean text:", empty_clean_text)

Empty clean text: 0


In [24]:
empty_clean_text = df_processed["clean_text"].str.strip().eq("").sum()

print("Empty clean text:", empty_clean_text)

Empty clean text: 0


In [25]:
clean_duplicate_groups = (
    df_processed
    .groupby("clean_text")
    .size()
    .sort_values(ascending=False)
)

print(
    "Repeated clean-text groups:",
    (clean_duplicate_groups > 1).sum()
)

Repeated clean-text groups: 4514


In [26]:
print(
    "Rows belonging to repeated clean-text groups:",
    clean_duplicate_groups[clean_duplicate_groups > 1].sum()
)

Rows belonging to repeated clean-text groups: 9028


## 2.6 Validate Labels for Repeated Clean Text

After text normalization, some tickets may have identical cleaned text.

Before creating the train, validation, and test sets, we check whether repeated
cleaned tickets have consistent target labels.

This is important because identical text with different labels could indicate
ambiguous data or potential data-quality issues.

The target variables in this project are:

- `type`
- `priority`
- `queue`

In [27]:
label_consistency = (
    df_processed
    .groupby("clean_text")
    .agg(
        count=("clean_text", "size"),
        type_unique=("type", "nunique"),
        priority_unique=("priority", "nunique"),
        queue_unique=("queue", "nunique")
    )
)

repeated_clean_text = label_consistency[
    label_consistency["count"] > 1
]

print("Repeated clean-text groups:", len(repeated_clean_text))

print(
    "Groups with different type labels:",
    (repeated_clean_text["type_unique"] > 1).sum()
)

print(
    "Groups with different priority labels:",
    (repeated_clean_text["priority_unique"] > 1).sum()
)

print(
    "Groups with different queue labels:",
    (repeated_clean_text["queue_unique"] > 1).sum()
)

Repeated clean-text groups: 4514
Groups with different type labels: 0
Groups with different priority labels: 0
Groups with different queue labels: 0


## 2.7 Leakage-Safe Train, Validation, and Test Split

The dataset contains repeated ticket text. Therefore, a simple random row-level
train-test split could place identical tickets in both the training and test sets,
leading to information leakage.

To prevent this, `clean_text` is used as the grouping variable. All rows containing
the same cleaned ticket text are kept in the same split.

The data is divided into:

- 70% training data
- 15% validation data
- 15% test data

The training set will be used to train the models, the validation set will be used
for model selection and tuning, and the test set will be used only for final
evaluation.

In [28]:
from sklearn.model_selection import GroupShuffleSplit

In [29]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    gss.split(
        df_processed,
        groups=df_processed["clean_text"]
    )
)

train_df = df_processed.iloc[train_idx].copy()
temp_df = df_processed.iloc[temp_idx].copy()

In [30]:
gss_temp = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss_temp.split(
        temp_df,
        groups=temp_df["clean_text"]
    )
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

In [31]:
print("Total:", len(df_processed))
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Total: 28261
Train: 19768
Validation: 4226
Test: 4267


In [32]:
total = len(df_processed)

print("Train %:", round(len(train_df) / total * 100, 2))
print("Validation %:", round(len(val_df) / total * 100, 2))
print("Test %:", round(len(test_df) / total * 100, 2))

Train %: 69.95
Validation %: 14.95
Test %: 15.1


In [33]:
train_texts = set(train_df["clean_text"])
val_texts = set(val_df["clean_text"])
test_texts = set(test_df["clean_text"])

print("Train ∩ Validation:", len(train_texts & val_texts))
print("Train ∩ Test:", len(train_texts & test_texts))
print("Validation ∩ Test:", len(val_texts & test_texts))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [34]:
for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}")
    print("\nType:")
    print(data["type"].value_counts(normalize=True).round(3))

    print("\nPriority:")
    print(data["priority"].value_counts(normalize=True).round(3))

    print("\nQueue:")
    print(data["queue"].value_counts(normalize=True).round(3))


Train

Type:
type
Incident    0.397
Request     0.289
Problem     0.209
Change      0.105
Name: proportion, dtype: float64

Priority:
priority
medium    0.414
high      0.381
low       0.205
Name: proportion, dtype: float64

Queue:
queue
Technical Support                  0.292
Product Support                    0.188
Customer Service                   0.149
IT Support                         0.115
Billing and Payments               0.105
Returns and Exchanges              0.050
Service Outages and Maintenance    0.039
Sales and Pre-Sales                0.029
Human Resources                    0.019
General Inquiry                    0.013
Name: proportion, dtype: float64

Validation

Type:
type
Incident    0.393
Request     0.288
Problem     0.207
Change      0.112
Name: proportion, dtype: float64

Priority:
priority
medium    0.396
high      0.389
low       0.215
Name: proportion, dtype: float64

Queue:
queue
Technical Support                  0.276
Product Support                  

In [35]:
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

print("\nLeakage check:")

train_texts = set(train_df["clean_text"])
val_texts = set(val_df["clean_text"])
test_texts = set(test_df["clean_text"])

print("Train ∩ Validation:", len(train_texts & val_texts))
print("Train ∩ Test:", len(train_texts & test_texts))
print("Validation ∩ Test:", len(val_texts & test_texts))

Train size: 19768
Validation size: 4226
Test size: 4267

Leakage check:
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


## 8. Save Preprocessed Dataset Splits

The leakage-safe train, validation, and test datasets are saved separately.
These files will be used in the modeling stage.

Saving the splits ensures that all models are evaluated on the same data partitions,
making the comparison between different approaches fair and reproducible.

In [36]:
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("Dataset splits saved successfully.")

Dataset splits saved successfully.
